# SESIÓN 2: Actividad práctica de PySpark

**Caso de negocio: análisis de pedidos y clientes**

El objetivo de esta actividad es analizar los pedidos y clientes utilizando PySpark en Databricks Free Edition. Se realizarán operaciones de agrupación, filtrado, agregación y unión de tablas para obtener información relevante para el negocio.

In [0]:
from pyspark.sql import functions as F

orders = spark.table("samples.tpch.orders")
customer = spark.table("samples.tpch.customer")
nation = spark.table("samples.tpch.nation")

print("Tablas cargadas correctamente")
print("Pedidos:", orders.count())
print("Clientes:", customer.count())
print("Países:", nation.count())

Tablas cargadas correctamente
Pedidos: 7500000
Clientes: 750000
Países: 25


# Ejercicio 1. Pedidos por estado

**Enunciado:** ¿Cuántos pedidos hay en cada estado (`o_orderstatus`) y cuál es el monto total de cada uno?

**Explicación:** Se agrupan los pedidos por estado y se calcula el número de pedidos y la suma de sus importes.

In [0]:
pedidos_por_estado = (
    orders
    .groupBy("o_orderstatus")
    .agg(
        F.count("*").alias("numero_pedidos"),
        F.sum("o_totalprice").alias("monto_total")
    )
    .orderBy("o_orderstatus")
)

display(pedidos_por_estado)

o_orderstatus,numero_pedidos,monto_total
F,3654501,549181919365.27
O,3654310,548923692869.55
P,191189,35333603011.43


# Ejercicio 2. Evolución anual

**Enunciado:** ¿Cuántos pedidos y qué monto total hay por año? Los resultados deben ordenarse del año más antiguo al más reciente.

**Explicación:** Se extrae el año de la fecha de cada pedido y se agrupan los registros para calcular el número de pedidos y su importe total anual.

In [0]:
pedidos_por_anio = (
    orders
    .withColumn("anio", F.year("o_orderdate"))
    .groupBy("anio")
    .agg(
        F.count("*").alias("numero_pedidos"),
        F.sum("o_totalprice").alias("monto_total")
    )
    .orderBy("anio")
)

display(pedidos_por_anio)

anio,numero_pedidos,monto_total
1992,1139873,172177707475.98
1993,1138893,172229906369.71
1994,1137944,172079589599.59
1995,1136308,171632673720.20
1996,1141556,172605588830.66
1997,1137325,171861381581.98
1998,668101,100852367668.13


# Ejercicio 3. Pedidos grandes ya finalizados

**Enunciado:** ¿Cuántos pedidos finalizados (`o_orderstatus = 'F'`) tienen un importe superior a 100.000?

**Explicación:** Se filtran los pedidos que cumplen simultáneamente ambas condiciones y se cuenta el número total de registros obtenidos.

In [0]:
pedidos_grandes_finalizados = (
    orders
    .filter(
        (F.col("o_orderstatus") == "F") &
        (F.col("o_totalprice") > 100000)
    )
)

resultado_pedidos_grandes = (
    pedidos_grandes_finalizados
    .agg(
        F.count("*").alias("numero_pedidos")
    )
)

display(resultado_pedidos_grandes)

numero_pedidos
2418589


# Ejercicio 4. Clientes por segmento

**Enunciado:** ¿Cuántos clientes hay en cada segmento de mercado (`c_mktsegment`)? Los resultados deben ordenarse de mayor a menor.

**Explicación:** Se agrupan los clientes por segmento de mercado, se cuenta cuántos pertenecen a cada uno y se ordenan los resultados de forma descendente.

In [0]:
clientes_por_segmento = (
    customer
    .groupBy("c_mktsegment")
    .agg(
        F.count("*").alias("numero_clientes")
    )
    .orderBy(F.desc("numero_clientes"))
)

display(clientes_por_segmento)

c_mktsegment,numero_clientes
BUILDING,150297
FURNITURE,150190
MACHINERY,150165
HOUSEHOLD,149687
AUTOMOBILE,149661


## Ejercicio 5. Clientes por país

**Enunciado:** ¿En qué cinco países (`n_name`) hay más clientes? Se debe mostrar el país y el número de clientes, ordenados de mayor a menor.

**Explicación:** Se unen las tablas de clientes y países mediante sus claves correspondientes. Después, se agrupan los clientes por país, se cuentan y se seleccionan los cinco países con más clientes.

In [0]:
clientes_por_pais = (
    customer
    .join(
        nation,
        customer["c_nationkey"] == nation["n_nationkey"],
        "inner"
    )
    .groupBy("n_name")
    .agg(
        F.count("*").alias("numero_clientes")
    )
    .orderBy(F.desc("numero_clientes"))
    .limit(5)
)

display(clientes_por_pais)

n_name,numero_clientes
INDONESIA,30401
IRAN,30257
INDIA,30234
IRAQ,30232
ETHIOPIA,30201


# Ejercicio 6. Ticket promedio por segmento

**Enunciado:** ¿Cuál es el ticket promedio (importe medio por pedido, `o_totalprice`) de cada segmento de mercado? Los resultados deben ordenarse de mayor a menor.

**Explicación:** Se unen las tablas de pedidos y clientes mediante sus claves. Después, se agrupan los pedidos por segmento de mercado y se calcula el importe medio de los pedidos de cada segmento.

In [0]:
ticket_promedio_por_segmento = (
    orders
    .join(
        customer,
        orders["o_custkey"] == customer["c_custkey"],
        "inner"
    )
    .groupBy("c_mktsegment")
    .agg(
        F.avg("o_totalprice").alias("ticket_promedio")
    )
    .orderBy(F.desc("ticket_promedio"))
)

display(ticket_promedio_por_segmento)

c_mktsegment,ticket_promedio
FURNITURE,151162.894836
BUILDING,151145.716568
MACHINERY,151127.451957
HOUSEHOLD,151112.323379
AUTOMOBILE,151077.375533


# Conclusiones

En esta actividad se han aplicado diferentes operaciones de PySpark para analizar los datos de pedidos y clientes mediante Databricks. Se han utilizado agrupaciones (`groupBy`), filtros (`filter`), agregaciones (`count`, `sum` y `avg`), uniones de tablas (`join`) y ordenación de resultados (`orderBy`).

Estas operaciones permiten obtener información relevante sobre los pedidos, la evolución anual de las ventas, los segmentos de mercado y la distribución de clientes por país. De esta forma, se demuestra la utilidad de PySpark para procesar y analizar grandes volúmenes de datos de manera distribuida.